# 02｜函數與模型：用程式描述身體的變化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wang3yuan/calculus-for-iih/blob/main/notebooks/02_functions_models.ipynb)

臺北醫學大學 智慧醫療跨領域　微積分（1151）　**自由練習，不計分、不用繳交**

對應課堂投影片：**Calculus02 函數與模型**

**怎麼用**
1. 按上方選單「執行階段 → 全部執行」，從上往下看圖和數字。
2. 看到 **✏️ 動手** 的格子，就自己改改看。填空沒填也不會出錯，程式會提醒你「還沒填」。
3. 卡住就點開「💡 提示」，真的不行再看「參考答案」。

不需要先會寫 Python，每一段程式前面都有中文說明。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def check(ok, hint):
    """檢查答案：對了顯示 ✓，不對顯示提示。"""
    print("✓ 正確！" if ok else "✗ 還沒對，" + hint)

## 1. 函數就是「輸入 → 輸出」

靜脈輸液以固定速率 125 mL/hr 進行，t 小時後的總輸入量是 $V(t)=125t$。在 Python 裡，`def` 就是在定義一個函數。

In [ ]:
def V(t):
    return 125 * t      # 線性模型：變化率固定

for t in [0, 1, 2, 8]:
    print(f"{t} 小時後，輸入 {V(t)} mL")

## 2. 一份虛擬病人資料

下面用亂數產生 12 位**虛擬**病人。`seed` 固定，所以每個人跑出來都一樣。

In [ ]:
rng = np.random.default_rng(2026)
patients = pd.DataFrame({
    "id": [f"P{i:02d}" for i in range(1, 13)],
    "height_cm": rng.normal(165, 8, 12).round(0),
    "weight_kg": rng.normal(62, 11, 12).round(1),
    "eGFR": rng.uniform(10, 105, 12).round(0),
})
patients

## 3. ✏️ 動手 1：分段函數

依腎功能調整劑量係數（示意規則，非用藥建議）：

$$A(\mathrm{eGFR})=\begin{cases}100\%, & \mathrm{eGFR}\ge 60\\ 75\%, & 30\le \mathrm{eGFR}<60\\ 50\%, & 15\le \mathrm{eGFR}<30\\ \text{個別評估}, & \mathrm{eGFR}<15\end{cases}$$

`if`／`elif` 就是分段函數的「分段」。請補上第三段的回傳值。

In [ ]:
def dose_factor(egfr):
    if egfr >= 60:
        return 1.0
    elif egfr >= 30:
        return 0.75
    elif egfr >= 15:
        return None      # ✏️ 填空：這一段的係數是多少？（用小數）
    else:
        return np.nan   # nan 表示「個別評估」，不給固定係數

# ---- 自動檢查（不用改）----
if dose_factor(20) is None:
    print("還沒填，把 None 換成這一段的係數。")
else:
    check(dose_factor(20) == 0.5, "看上面的公式：15 ≤ eGFR < 30 時係數是 50%。")
    patients["factor"] = patients["eGFR"].apply(dose_factor)
    print(patients[["id", "eGFR", "factor"]].to_string(index=False))

## 4. ✏️ 動手 2：冪函數與體表面積

Mosteller 公式 $\mathrm{BSA}=\sqrt{h\times w/3600}$（h 是身高 cm、w 是體重 kg）。開根號就是 $\tfrac12$ 次方，在 Python 寫成 `** 0.5`。

In [ ]:
def bsa(h, w):
    return None      # ✏️ 填空：把公式寫出來

# ---- 自動檢查（不用改）----
if bsa(160, 60) is None:
    print("還沒填，把 None 換成公式。")
else:
    check(abs(bsa(160, 60) - 1.633) < 0.01, "(h * w / 3600) 外面要開根號。")
    print(f"身高 160、體重 60 的 BSA ≈ {bsa(160, 60):.2f} m²")
    print(f"若藥物建議 100 mg/m²，劑量約 {100 * bsa(160, 60):.0f} mg")

<details><summary>參考答案</summary>

`return (h * w / 3600) ** 0.5`
</details>

## 5. 指數衰減與半衰期

藥物濃度 $C(t)=C_0e^{-kt}$，半衰期 $t_{1/2}=\ln 2/k$。下面假設 $C_0=100$ mg/L、半衰期 4 小時，並產生**虛擬**量測值（加上一點雜訊）。

左圖是一般座標，右圖把 y 軸改成對數：**指數衰減在對數座標上會變成直線**。

In [ ]:
half_life = 4
k = np.log(2) / half_life
t = np.linspace(0, 24, 200)
C = 100 * np.exp(-k * t)

rng = np.random.default_rng(7)
t_obs = np.arange(0, 25, 3)
C_obs = 100 * np.exp(-k * t_obs) * np.exp(rng.normal(0, 0.06, t_obs.size))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a in ax:
    a.plot(t, C, color="#028090", lw=3, label="model")
    a.plot(t_obs, C_obs, "o", color="#D9534F", label="virtual data")
    a.set_xlabel("t (hours)"); a.set_ylabel("C (mg/L)"); a.grid(alpha=0.3); a.legend()
ax[1].set_yscale("log")
ax[0].set_title("linear axis"); ax[1].set_title("log axis")
plt.tight_layout(); plt.show()

## 6. ✏️ 動手 3：五個半衰期之後

每過一個半衰期，濃度剩一半。請算出經過 **5 個半衰期**後，還剩原本的百分之幾。

In [ ]:
answer_3 = None      # ✏️ 填空：剩下百分之幾？（可以寫算式，例如 100 * 0.5 ** 2）

# ---- 自動檢查（不用改）----
if answer_3 is None:
    print("還沒填，把 None 換成你的答案。")
else:
    check(abs(answer_3 - 3.125) < 0.01, "每個半衰期乘一次 0.5，要乘 5 次。")

## 7. 對數：把很大的範圍壓小

$\mathrm{pH}=-\log_{10}[\mathrm{H}^+]$。pH 差 1，氫離子濃度就差 10 倍。

In [ ]:
for H in [1e-2, 1e-4, 1e-7, 10**-7.4, 1e-9]:
    print(f"[H⁺] = {H:.2e} mol/L   pH = {-np.log10(H):.2f}")

## 8. 週期函數與函數的操作

體溫有晝夜節律，可用 $T(t)=36.8+0.4\sin\!\big(\pi(t-10)/12\big)$ 描述（t 是小時，示意數值）。

- `36.8`：上下平移（平均體溫）
- `0.4`：振幅
- `t - 10`：左右平移
- `π/12`：決定週期（這裡是 24 小時）

**✏️ 動手 4**：改下面的數字，看曲線怎麼變。例如把 `amp` 改成 `0.8`，或把 `shift` 改成 `0`。

In [ ]:
amp = 0.4      # ← 改這裡（振幅）
shift = 10     # ← 改這裡（左右平移）

t = np.linspace(0, 48, 400)
T = 36.8 + amp * np.sin(np.pi * (t - shift) / 12)

plt.figure(figsize=(8, 3.5))
plt.plot(t, T, color="#028090", lw=3)
plt.xlabel("t (hours)"); plt.ylabel("T (°C)"); plt.grid(alpha=0.3)
plt.show()

## 9. 用程式檢查 AI 的答案

> AI 說：「半衰期 4 小時，所以 12 小時是 3 個半衰期，藥物剩下原本的 1/3。」

用模型算算看。

In [ ]:
print("AI 說剩下：", round(100 / 3, 1), "%")
print("模型算出 ：", round(100 * np.exp(-k * 12), 1), "%")

AI 把「每次減半」當成「平均分成 3 份」。3 個半衰期是 $0.5^3=12.5\%$。**指數變化是連乘，不是平分。**

## 小結

- 函數是輸入到輸出的規則；`def` 就是定義函數，`if` 就是分段。
- 線性是「固定的量」，指數是「固定的比例」。
- 對數座標把指數變化畫成直線，也把很大的範圍壓小。

---
*本 notebook 為教學練習，數值與情境皆為示意，資料為程式產生的虛擬資料，非真實病人資料，也不是用藥建議。*